In [ ]:
using Bessels, Plots, LinearAlgebra

const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
φ_b = 0.204

E_J = 37.12*2pi/2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 4*g2

g = E_J*sin(ε_p)*φ_a^2*φ_b
ε_d_ratio = 2
params = (ε_p,E_J,ε_d_ratio,κ_b);

M = 100

A_0 = sqrt(ε_d / g)
B_0 = sqrt(2) * ε_d / κ_b

In [ ]:
using NLsolve

function H(A, B, P_A, P_B, params)
    (ε_p,E_J,ε_d_ratio,κ_b) = params
    g = E_J*sin(ε_p)*φ_a^2*φ_b
    ε_d = ε_d_ratio*g
    return -sqrt(2)*g*A*P_A*B +
           (g*(A^2 + P_A^2)/sqrt(2) - ε_d/sqrt(2) - κ_b*B/2)*P_B -
           κ_b*P_B^2/2
end

function gradH(A, B, P_A, P_B, params)
    (ε_p,E_J,ε_d_ratio,κ_b) = params
    g = E_J*sin(ε_p)*φ_a^2*φ_b
    ε_d = ε_d_ratio*g
    dA   = -sqrt(2)*g*(P_A*B-A*P_B)
    dB   = -(κ_b/2)*P_B-sqrt(2)*g*A*P_A
    dP_A = -sqrt(2)*g*A*B + sqrt(2)*g*P_A*P_B
    dP_B = g*(A^2 + P_A^2)/sqrt(2) - ε_d/sqrt(2) - κ_b*B/2 - κ_b*P_B
    return dA, dB, dP_A, dP_B
end
function step_diabatic(A_init, B_init, P_A_init, P_B_init, params)

    A_proj   = similar(A_init)
    B_proj   = similar(B_init)
    P_A_proj = similar(P_A_init)
    P_B_proj = similar(P_B_init)
    dist_proj = similar(A_init)

    for i in eachindex(A_init)
        A0   = A_init[i]
        B0   = B_init[i]
        P_A0 = P_A_init[i]
        P_B0 = P_B_init[i]

        function F!(F, x)
            A, B, P_A, P_B, λ = x
            dA, dB, dP_A, dP_B = gradH(A, B, P_A, P_B, params)

            F[1] = A   - A0   + λ*dA
            F[2] = B   - B0   + λ*dB
            F[3] = P_A - P_A0 + λ*dP_A
            F[4] = P_B - P_B0 + λ*dP_B
            F[5] = H(A, B, P_A, P_B, params)
        end

        x0 = [A0, B0, P_A0, P_B0, 0.0]

        sol = nlsolve(F!, x0)

        A_proj[i]   = sol.zero[1]
        B_proj[i]   = sol.zero[2]
        P_A_proj[i] = sol.zero[3]
        P_B_proj[i] = sol.zero[4]

        dist_proj[i] = sqrt(
            (A_proj[i]   - A0)^2 +
            (B_proj[i]   - B0)^2 +
            (P_A_proj[i] - P_A0)^2 +
            (P_B_proj[i] - P_B0)^2
        )
    end
    return A_proj, B_proj, P_A_proj, P_B_proj
end

In [ ]:
ϕ_a = [A_0, 0]
ϕ_b = [0, -B_0]

# initial string in phase-space
Δs = 1 / M
s=0:Δs:1

A_init = [A_0*(1 - s_i) for s_i in s]
P_A_init = [A_0*s_i for s_i in s]
P_B_init = [2 * A_init[i]/A_0 * (A_init[i]/A_0-1)*B_0 for (i, s_i) in enumerate(s)]
B_init = [-P_B_init[i] for (i, s_i) in enumerate(s)]

ϕ_init = hcat(A_init, B_init)
θ_init = hcat(P_A_init, P_B_init);

In [ ]:
ε_p_list = LinRange(0.1, 1.1, 10)
for ε_p in ε_p_list
    p0 = plot()
    params = (ε_p,E_J,ε_d_ratio,κ_b)
    println(params)
    A_init, B_init, P_A_init, P_B_init = step_diabatic(A_init, B_init, P_A_init, P_B_init, params)
    scatter!(p0, P_B_init, A_init)
    display(p0)
end
ε_p = ε_p_list[end]

In [ ]:
E_J_list = LinRange(37*2*pi/2, 37*2*pi*1.5, 10)
for E_J in E_J_list
    p0 = plot()
    params = (ε_p,E_J,ε_d_ratio,κ_b)
    println(params)
    A_init, B_init, P_A_init, P_B_init = step_diabatic(A_init, B_init, P_A_init, P_B_init, params)
    scatter!(p0, P_B_init, A_init)
    display(p0)
end
E_J = E_J_list[end]

In [ ]:
κ_b_list = LinRange(5/10.4, 1/10.4, 10)
for κ_b in κ_b_list
    p0 = plot()
    params = (ε_p,E_J,ε_d_ratio,κ_b)
    println(params)
    A_init, B_init, P_A_init, P_B_init = step_diabatic(A_init, B_init, P_A_init, P_B_init, params)
    scatter!(p0, P_B_init, A_init)
    display(p0)
end
κ_b = κ_b_list[end]

In [ ]:
ε_d_ratio_list = LinRange(2, 8, 10)
for ε_d_ratio in ε_d_ratio_list
    p0 = plot()
    params = (ε_p,E_J,ε_d_ratio,κ_b)
    println(params)
    A_init, B_init, P_A_init, P_B_init = step_diabatic(A_init, B_init, P_A_init, P_B_init, params)
    scatter!(p0, P_B_init, A_init)
    display(p0)
end
ε_d_ratio = ε_d_ratio_list[end]

In [ ]:
# --- subplot 1 ---
p1 = scatter(P_A_init, A_init, ms=.5,
    xlabel="P_A",
    ylabel="A",
     label="GDA")

# --- subplot 2 ---
p2 = scatter(P_B_init, B_init, ms=.5,
    xlabel="P_B",
    ylabel="B",
    label="GDA")

# --- subplot 3 ---
p3 = scatter(A_init, B_init, ms=.5,
    xlabel="A",
    ylabel="B",
    label="GDA")

# --- combine ---
plot(p1, p2, p3, layout=(1,3))

In [ ]:
scatter(A_init)
scatter!(P_A_init)

In [ ]:
scatter(B_init)
scatter!(P_B_init)

In [ ]:
points = [[A_init[i], B_init[i], P_A_init[i], P_B_init[i]] for i in eachindex(A_init)]
params = (ε_p,E_J,ε_d_ratio,κ_b)
H_check = [H(A_init[i], B_init[i], P_A_init[i], P_B_init[i], params) for i in eachindex(A_init)]
plot(H_check)

In [ ]:
s = zeros(M + 1)
for i in 2:M+1
    d1 = points[i][1] - points[i-1][1]
    d2 = points[i][2] - points[i-1][2]
    d3 = points[i][3] - points[i-1][3]
    d4 = points[i][4] - points[i-1][4]
    s[i] = s[i-1] + sqrt(d1^2 + d2^2 + d3^2 + d4^2)
end
stringLength = s[end]
for i in 2:M+1
    s[i] /= stringLength
end
normalizedString = collect(range(0.0, 1.0, length=M+1))

points_renorm = copy(points)
for i in 2:M
    lastPoint = 2
    for k in lastPoint:M+1
        if s[k-1] <= normalizedString[i] < s[k]
            q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])
            points_renorm[i] .= (1 - q) .* points[k-1] .+ q .* points[k]
            break
        end
        lastPoint = k
    end
end

In [ ]:
M = length(points_renorm)

T_list = Vector{Vector{Float64}}(undef, M-2)
F_list = Vector{Vector{Float64}}(undef, M-2)
lambda_list = fill(NaN, M-2)
residual_list = Vector{Vector{Float64}}(undef, M-2)

for i in 2:M-1
    # discrete tangent
    T = points_renorm[i+1] .- points_renorm[i-1]

    # Hamiltonian flow at node i
    A, B, P_A, P_B = points_renorm[i]
    dA, dB, dP_A, dP_B = gradH(A, B, P_A, P_B, params)
    F = [dP_A, dP_B, -dA, -dB]

    T_list[i-1] = T
    F_list[i-1] = F

    # best local proportionality factor
    denom = dot(F, F)
    if denom > 0.0
        λ = dot(T, F) / denom
    else
        λ = NaN
    end
    lambda_list[i-1] = λ

    # residual: T - λ F
    residual_list[i-1] = T .- λ .* F
end
relerr = fill(NaN, M-2)
for i in 1:M-2
    relerr[i] = norm(residual_list[i, :]) / (norm(T_list[i, :]) + norm(F_list[i, :]) + 1e-14)
end

In [ ]:
plot(relerr)